# Audit kesetaraan modul skrining hs-CRP 5 butir (untuk naskah Health Systems)
Pertanyaan: rata-rata nasional (AUC 0,68; O/E 1,00) tampak baik — **siapa yang paling banyak terlewat?**
Kelompok: jenis kelamin, umur, desa/kota, pengeluaran, wilayah. Ambang rujukan 0,155 (sama dengan model card).
Dijalankan **di komputer sendiri**; yang dibagikan ke Claude hanya tabel ringkasan (Ctrl+S).
Waktu: ±10–20 menit (bootstrap 500×).

In [ ]:
import os, time, warnings
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")

ROOT = r"F:\SINTA 3\crp-ifls5-ml"
OUT = os.path.join(ROOT, "results")
DATA_FILE = os.path.join(ROOT, "data", "crp_ifls5_model.csv")
SEED, REP_OOF, N_BOOT, THR = 42, 3, 500, 0.155
FIVE = ["bmi", "female", "age", "srh_poor", "urban"]
rng = np.random.default_rng(SEED)

df = pd.read_csv(DATA_FILE, dtype={"pidlink": str, "hhid14": str})
y, g = df.crp_high.values.astype(int), df.hhid14.values
X = df[FIVE].astype(float)
prov = pd.to_numeric(df.province, errors="coerce")
REGION = {1: "Sumatra", 3: "Java", 5: "Bali–Nusa Tenggara", 6: "Kalimantan", 7: "Sulawesi"}
q = pd.qcut(df.lnpce.rank(method="first"), 5, labels=False) + 1
GROUPS = {
    "Sex": df.female.map({0: "Male", 1: "Female"}),
    "Age group": pd.cut(df.age, [0, 39, 59, 200], labels=["15–39", "40–59", "≥60"]).astype(str),
    "Residence": df.urban.map({0: "Rural", 1: "Urban"}),
    "Expenditure": q.map({1: "Q1–Q2 (poorest)", 2: "Q1–Q2 (poorest)", 3: "Q3", 4: "Q4–Q5 (richest)", 5: "Q4–Q5 (richest)"}),
    "Region": (prov // 10).map(REGION).fillna("Other"),
}
REF = {"Sex": "Female", "Age group": "15–39", "Residence": "Urban", "Expenditure": "Q4–Q5 (richest)", "Region": "Java"}
print("n =", len(df), "| kasus =", y.sum(), f"({y.mean()*100:.1f}%)")
for k, v in GROUPS.items():
    print(k, v.value_counts().to_dict())

## 1. Prediksi out-of-fold terkalibrasi (sama persis dengan naskah sebelumnya)

In [ ]:
def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))
def model():
    return Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler()),
                     ("m", LogisticRegression(class_weight="balanced", max_iter=2000))])
def fit_platt(p, yt):
    lr = LogisticRegression(C=1e6, max_iter=1000).fit(logit(p).reshape(-1, 1), yt)
    return lambda s: lr.predict_proba(logit(s).reshape(-1, 1))[:, 1]

t0 = time.time()
P = np.zeros((REP_OOF, len(y))); FOLDS = []
for r in range(REP_OOF):
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED + r)
    for tr, te in cv.split(X, y, g):
        m = model().fit(X.iloc[tr], y[tr])
        inner = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED)
        pin = np.zeros(len(tr))
        for a, b in inner.split(X.iloc[tr], y[tr], g[tr]):
            pin[b] = model().fit(X.iloc[tr].iloc[a], y[tr][a]).predict_proba(X.iloc[tr].iloc[b])[:, 1]
        cal = fit_platt(pin, y[tr])
        P[r, te] = cal(m.predict_proba(X.iloc[te])[:, 1])
        FOLDS.append((r, tr, te, cal(pin)))      # simpan prediksi latih (OOF dalam) untuk mitigasi
p = P.mean(0)
print(f"AUC keseluruhan = {roc_auc_score(y, p):.3f} | O/E = {y.mean()/p.mean():.3f} | {(time.time()-t0)/60:.1f} menit")

## 2. Kinerja per kelompok pada ambang nasional 0,155 (CI bootstrap per rumah tangga)

In [ ]:
def citl_slope(yt, pp):
    lp = logit(pp)
    slope = LogisticRegression(C=1e6, max_iter=1000).fit(lp.reshape(-1, 1), yt).coef_[0][0]
    citl = brentq(lambda a: (1 / (1 + np.exp(-(lp + a)))).mean() - yt.mean(), -10, 10)
    return citl, slope

def stats(yt, pp, t=THR):
    f = pp >= t; tp = (f & (yt == 1)).sum(); fn = (~f & (yt == 1)).sum()
    out = {"n": len(yt), "Cases": int(yt.sum()), "Prevalence (%)": yt.mean() * 100,
           "AUC": roc_auc_score(yt, pp) if 0 < yt.sum() < len(yt) else np.nan,
           "Sensitivity": tp / max(yt.sum(), 1), "Missed cases (FNR, %)": fn / max(yt.sum(), 1) * 100,
           "Flagged (%)": f.mean() * 100, "PPV": tp / max(f.sum(), 1),
           "Tests per 1000": f.mean() * 1000, "Cases found per 1000": tp / len(yt) * 1000,
           "Cases missed per 1000": fn / len(yt) * 1000, "O/E": yt.mean() / pp.mean(),
           "Mean predicted risk in cases": pp[yt == 1].mean()}
    try:
        out["Calibration intercept"], out["Calibration slope"] = citl_slope(yt, pp)
    except Exception:
        out["Calibration intercept"], out["Calibration slope"] = np.nan, np.nan
    return out

HH = pd.Series(g); uniq = HH.unique(); idx_by_hh = HH.groupby(HH).indices
BOOT = [np.concatenate([idx_by_hh[h] for h in rng.choice(uniq, len(uniq))]) for _ in range(N_BOOT)]

rows, gaps = [], []
KEYS = ["AUC", "Missed cases (FNR, %)", "Flagged (%)", "O/E", "Calibration intercept"]
for attr, lab in GROUPS.items():
    lab = lab.values
    for grp in [x for x in pd.unique(lab) if x not in ("Other", "nan")]:
        m = lab == grp
        est = stats(y[m], p[m]); bs = []
        for b in BOOT[:300]:
            bb = b[m[b]]
            if 0 < y[bb].sum() < len(bb): bs.append(stats(y[bb], p[bb]))
        bs = pd.DataFrame(bs)
        row = {"Attribute": attr, "Group": grp, **{k: round(v, 3) for k, v in est.items()}}
        for k in KEYS:
            lo, hi = bs[k].quantile([.025, .975]); row[f"{k} 95% CI"] = f"{lo:.2f}–{hi:.2f}"
        rows.append(row)
    # selisih FNR terhadap kelompok rujukan
    ref = REF[attr]
    for grp in [x for x in pd.unique(lab) if x not in ("Other", "nan", ref)]:
        def gap(ix):
            a, r_ = ix[lab[ix] == grp], ix[lab[ix] == ref]
            fa = ((p[a] < THR) & (y[a] == 1)).sum() / max(y[a].sum(), 1)
            fr = ((p[r_] < THR) & (y[r_] == 1)).sum() / max(y[r_].sum(), 1)
            return (fa - fr) * 100
        allix = np.arange(len(y)); est = gap(allix); bs = np.array([gap(b) for b in BOOT])
        gaps.append({"Attribute": attr, "Group vs reference": f"{grp} vs {ref}", "FNR gap (pp)": round(est, 1),
                     "95% CI": f"{np.percentile(bs, 2.5):.1f} to {np.percentile(bs, 97.5):.1f}",
                     "p (two-sided, bootstrap)": round(2 * min((bs <= 0).mean(), (bs >= 0).mean()), 3)})
TAB = pd.DataFrame(rows); GAP = pd.DataFrame(gaps)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
print(TAB[["Attribute", "Group", "n", "Cases", "Prevalence (%)", "AUC", "Missed cases (FNR, %)", "Flagged (%)",
           "PPV", "O/E", "Calibration intercept", "Calibration slope"]].to_string(index=False))
print("\nSelisih kasus terlewat (FNR) terhadap kelompok rujukan:")
print(GAP.to_string(index=False))

## 3. Mengapa terlewat? Mekanisme (prevalensi vs kalibrasi vs nilai prediktor pada kasus)

In [ ]:
mech = []
for attr in ["Sex", "Residence", "Age group", "Region"]:
    lab = GROUPS[attr].values
    for grp in [x for x in pd.unique(lab) if x not in ("Other", "nan")]:
        m = (lab == grp) & (y == 1)
        mech.append({"Attribute": attr, "Group": grp, "Cases": int(m.sum()),
                     "Mean BMI in cases": df.bmi[m].mean(), "Share of cases with BMI <23 (%)": (df.bmi[m] < 23).mean() * 100,
                     "Mean age in cases": df.age[m].mean(), "Poor SRH in cases (%)": df.srh_poor[m].mean() * 100,
                     "Mean predicted risk in cases": p[m].mean(), "Cases below threshold (%)": (p[m] < THR).mean() * 100})
MECH = pd.DataFrame(mech).round(2)
print(MECH.to_string(index=False))

## 4. Mitigasi (dihitung lintas-fold; ambang/kalibrasi dipelajari dari data latih saja)
A. Ambang per jenis kelamin agar sensitivitas tiap kelompok = sensitivitas nasional di data latih.
B. Kalibrasi ulang intercept per wilayah (intercept update di data latih).
C. A + B.

In [ ]:
def thr_for_sens(pp, yt, target):
    s = np.sort(pp[yt == 1])[::-1]
    k = int(np.ceil(target * len(s))) - 1
    return s[max(min(k, len(s) - 1), 0)]

sex = GROUPS["Sex"].values; reg = GROUPS["Region"].values
PA, PB = np.full((REP_OOF, len(y)), np.nan), np.full((REP_OOF, len(y)), np.nan)
TA = np.full((REP_OOF, len(y)), np.nan)
for (r, tr, te, ptr) in FOLDS:
    target = ((ptr >= THR) & (y[tr] == 1)).sum() / y[tr].sum()
    # A: ambang per jenis kelamin
    for s_ in ["Female", "Male"]:
        mt = sex[tr] == s_
        TA[r, te[sex[te] == s_]] = thr_for_sens(ptr[mt], y[tr][mt], target)
    # B: intercept per wilayah
    adj = np.zeros(len(te))
    for rg in pd.unique(reg):
        mt, me = reg[tr] == rg, reg[te] == rg
        if mt.sum() > 50 and y[tr][mt].sum() > 5:
            a = brentq(lambda a: (1 / (1 + np.exp(-(logit(ptr[mt]) + a)))).mean() - y[tr][mt].mean(), -5, 5)
            adj[me] = a
    PB[r, te] = 1 / (1 + np.exp(-(logit(P[r, te]) + adj)))
    PA[r, te] = P[r, te]
thrA = np.nanmean(TA, 0); pB = np.nanmean(PB, 0)

def summary(name, flag, pp=None):
    out = {"Strategy": name, "Flagged (%)": flag.mean() * 100,
           "Sensitivity": (flag & (y == 1)).sum() / y.sum(), "Specificity": (~flag & (y == 0)).sum() / (y == 0).sum()}
    for s_ in ["Female", "Male"]:
        m = (sex == s_) & (y == 1); out[f"FNR {s_} (%)"] = (~flag[m]).mean() * 100
    out["FNR gap M−F (pp)"] = out["FNR Male (%)"] - out["FNR Female (%)"]
    for rg in ["Java", "Sumatra", "Bali–Nusa Tenggara", "Kalimantan", "Sulawesi"]:
        m = (reg == rg) & (y == 1); out[f"FNR {rg} (%)"] = (~flag[m]).mean() * 100
    fr = [out[f"FNR {rg} (%)"] for rg in ["Java", "Sumatra", "Bali–Nusa Tenggara", "Kalimantan", "Sulawesi"]]
    out["FNR range across regions (pp)"] = max(fr) - min(fr)
    if pp is not None: out["AUC"] = roc_auc_score(y, pp)
    return out

MIT = pd.DataFrame([
    summary("National rule (threshold 0.155)", p >= THR, p),
    summary("A. Sex-specific thresholds", p >= thrA, p),
    summary("B. Regional recalibration", pB >= THR, pB),
    summary("C. A + B", pB >= thrA, pB),
]).round(3)
print(MIT.T.to_string())

## 5. Sudut pandang sistem kesehatan: per 1.000 orang dewasa di tiap wilayah
Bandingkan "periksa semua" vs aturan nasional: tes yang dihemat, kasus ditemukan, kasus terlewat, tes per kasus ditemukan.
Skenario kapasitas: bila hanya 20% orang dewasa bisa dites, berapa kasus ditemukan per wilayah.

In [ ]:
sysrows = []
for rg in ["Java", "Sumatra", "Bali–Nusa Tenggara", "Kalimantan", "Sulawesi", "All"]:
    m = np.ones(len(y), bool) if rg == "All" else reg == rg
    f = p[m] >= THR; yt = y[m]; n = m.sum()
    cap = p[m] >= np.quantile(p[m], 0.80)        # 20% teratas di wilayah itu
    capn = p[m] >= np.quantile(p, 0.80)          # 20% teratas nasional (satu antrean nasional)
    sysrows.append({"Region": rg, "n": int(n), "Prevalence (%)": yt.mean() * 100,
                    "Tests avoided per 1000 vs test-all": (1 - f.mean()) * 1000,
                    "Cases found per 1000": (f & (yt == 1)).sum() / n * 1000,
                    "Cases missed per 1000": (~f & (yt == 1)).sum() / n * 1000,
                    "Tests per case found": f.sum() / max((f & (yt == 1)).sum(), 1),
                    "Test-all: tests per case": 1 / yt.mean(),
                    "20% capacity, regional ranking: cases found (%)": (cap & (yt == 1)).sum() / yt.sum() * 100,
                    "20% capacity, national ranking: share tested (%)": capn.mean() * 100,
                    "20% capacity, national ranking: cases found (%)": (capn & (yt == 1)).sum() / yt.sum() * 100})
SYS = pd.DataFrame(sysrows).round(2)
print(SYS.to_string(index=False))

## 6. Gambar dan simpan (agregat, aman dibagikan)

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 5.2))
order = TAB.copy(); order["lab"] = order.Attribute + ": " + order.Group
yy = np.arange(len(order))[::-1]
lo = order["Missed cases (FNR, %) 95% CI"].str.split("–").str[0].astype(float)
hi = order["Missed cases (FNR, %) 95% CI"].str.split("–").str[1].astype(float)
fnr = order["Missed cases (FNR, %)"]
ax.errorbar(fnr, yy, xerr=[fnr - lo, hi - fnr], fmt="o", color="#1f4e79", ecolor="#7f9fbf", capsize=2, ms=4)
overall = ((p < THR) & (y == 1)).sum() / y.sum() * 100
ax.axvline(overall, ls="--", color="grey", lw=1); ax.text(overall + 1, len(order) - 0.3, f"National {overall:.0f}%", fontsize=7, color="grey")
ax.set_yticks(yy); ax.set_yticklabels(order.lab, fontsize=7)
ax.set_xlabel("Adults with elevated hs-CRP missed by the national rule (%)")
ax.set_title("Missed cases by group at the national referral threshold", fontsize=9)
plt.tight_layout()
for ext in ["png", "tiff"]:
    plt.savefig(os.path.join(OUT, f"fig_audit_fnr.{ext}"), dpi=300)
plt.close()

with pd.ExcelWriter(os.path.join(OUT, "audit_kesetaraan_hsys.xlsx")) as w:
    TAB.to_excel(w, sheet_name="per_kelompok", index=False); GAP.to_excel(w, sheet_name="selisih_FNR", index=False)
    MECH.to_excel(w, sheet_name="mekanisme", index=False); MIT.to_excel(w, sheet_name="mitigasi", index=False)
    SYS.to_excel(w, sheet_name="sistem_kesehatan", index=False)
print("Tersimpan:", os.path.join(OUT, "audit_kesetaraan_hsys.xlsx"), "dan fig_audit_fnr.png/.tiff")
print("Simpan notebook (Ctrl+S) lalu kabari Claude.")